In [ ]:
!pip install -q google-generativeai

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "key"

In [ ]:
%%writefile prompt_library.py
"""
prompt_library.py
==================
Reusable, documented prompt templates for common LLM tasks.
"""

SUMMARIZE_PROMPT = """You are a precise summarization engine.

Summarize the following text in 1-2 sentences, and list up to 5 key points.

Text:
\"\"\"{text}\"\"\"

Respond with STRICT JSON ONLY, matching this exact schema, and nothing else
(no markdown fences, no explanations):
{{
  "summary": "<string>",
  "key_points": ["<string>", "..."]
}}"""

CLASSIFY_PROMPT = """You are a strict text classifier.

Classify the following text into exactly ONE of these categories:
{categories}

Text:
\"\"\"{text}\"\"\"

Respond with STRICT JSON ONLY, matching this exact schema, and nothing else
(no markdown fences, no explanations):
{{
  "category": "<string, must be one of the given categories>",
  "confidence": <number between 0 and 1>,
  "reasoning": "<short string>"
}}"""

EXTRACT_PROMPT = """You are an information-extraction engine.

From the text below, extract these fields: {fields}
If a field is not present, set its value to null and list it under "missing_fields".

Text:
\"\"\"{text}\"\"\"

Respond with STRICT JSON ONLY, matching this exact schema, and nothing else:
{{
  "extracted": {{"<field_name>": "<value or null>"}},
  "missing_fields": ["<string>", "..."]
}}"""

SENTIMENT_PROMPT = """You are a sentiment analysis engine.

Analyze the sentiment of the following text.

Text:
\"\"\"{text}\"\"\"

Respond with STRICT JSON ONLY, matching this exact schema, and nothing else:
{{
  "sentiment": "positive" | "negative" | "neutral",
  "score": <number between -1 and 1>,
  "emotions": ["<string>", "..."]
}}"""

PROMPT_LIBRARY = {
    "summarize": SUMMARIZE_PROMPT,
    "classify": CLASSIFY_PROMPT,
    "extract": EXTRACT_PROMPT,
    "sentiment": SENTIMENT_PROMPT,
}

Writing prompt_library.py


In [ ]:
%%writefile cli_qa_tool.py
"""
cli_qa_tool.py
==============
A command-line question-answering tool using the Gemini API.
Reuses the Day 9 API-call pattern and the Day 12 prompt library.
Handles empty or weird input without crashing.
"""

import os

from prompt_library import PROMPT_LIBRARY  # reused from Day 12

SYSTEM_PROMPT = (
    "You are a clear, concise, and helpful assistant. "
    "Answer the user's question directly. If the question is empty, "
    "unclear, or not really a question, politely ask the user to rephrase "
    "instead of guessing."
)


def call_model(question: str) -> str:
    """Send the question to Gemini and return the text reply. Falls back
    to a mock reply if no API key is set, so the tool never crashes."""
    if not os.environ.get("GEMINI_API_KEY"):
        return f"[MOCK REPLY - no API key set] You asked: {question!r}"

    import google.generativeai as genai

    genai.configure(api_key=os.environ["GEMINI_API_KEY"])
    model = genai.GenerativeModel(
        model_name="gemini-2.0-flash",
        system_instruction=SYSTEM_PROMPT,
    )
    response = model.generate_content(question)
    return response.text


def ask(question: str) -> str:
    """Wraps the model call in try/except so bad input or API errors
    never crash the program."""
    question = (question or "").strip()

    if not question:
        return "Please type an actual question — I got nothing to work with."

    try:
        return call_model(question)
    except Exception as e:
        return f"Sorry, something went wrong while getting an answer: {e}"


def main():
    print("=== CLI Question-Answering Tool (Gemini) ===")
    print("Type a question and press Enter. Type 'quit' or 'exit' to stop.\n")

    while True:
        try:
            question = input("Q: ")
        except (EOFError, KeyboardInterrupt):
            print("\nExiting.")
            break

        if question.strip().lower() in ("quit", "exit"):
            print("Exiting.")
            break

        answer = ask(question)
        print(f"A: {answer}\n")


if __name__ == "__main__":
    main()

Writing cli_qa_tool.py


In [ ]:
!python3 cli_qa_tool.py

=== CLI Question-Answering Tool (Gemini) ===
Type a question and press Enter. Type 'quit' or 'exit' to stop.

Q: What is RAG?
/content/cli_qa_tool.py:27: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
A: Sorry, something went wrong while getting an answer: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.goo